# Nexora test fixtures

Creates small synthetic tables in schema `nexora_test` of the **sandbox** Lakehouse. They deliberately break rules (duplicate dimension keys, a many-to-many relationship with and without a mapping table) and add a physical date table, so the acceptance scenarios can be tested. See `fixtures/README.md`.

* Attach the **sandbox** Lakehouse (schemas enabled) as this notebook's default Lakehouse before running.
* Writes only to schema `nexora_test`. Never writes to `saleslt`.
* Rerunnable: every table is overwritten with identical content.

In [ ]:
SCHEMA = "nexora_test"        # the only schema this notebook ever writes to
SOURCE_SCHEMA = "saleslt"     # read only

try:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")
except Exception as error:
    raise RuntimeError(
        f"Could not create schema '{SCHEMA}'. The fixtures need a schema-enabled "
        "Lakehouse: attach the sandbox Lakehouse (created with schemas enabled) "
        "as the default Lakehouse of this notebook."
    ) from error

try:
    spark.table(f"{SOURCE_SCHEMA}.SalesLT_Product").limit(1).collect()
    spark.table(f"{SOURCE_SCHEMA}.SalesLT_SalesOrderHeader").limit(1).collect()
except Exception as error:
    raise RuntimeError(
        f"Could not read {SOURCE_SCHEMA}.SalesLT_Product / SalesLT_SalesOrderHeader. "
        "Attach the Lakehouse that holds the saleslt sample data."
    ) from error

In [ ]:
PRODUCT_COUNT = 12

# Real ProductIDs, chosen deterministically so every run gives the same tables.
product_ids = [
    row["ProductID"]
    for row in spark.table(f"{SOURCE_SCHEMA}.SalesLT_Product")
    .select("ProductID").distinct().orderBy("ProductID").limit(PRODUCT_COUNT).collect()
]

if len(product_ids) < PRODUCT_COUNT:
    raise RuntimeError(
        f"Need at least {PRODUCT_COUNT} products in {SOURCE_SCHEMA}.SalesLT_Product, "
        f"found {len(product_ids)}."
    )

In [ ]:
import calendar
from datetime import date, timedelta

CATEGORIES = ["Bikes", "Components", "Clothing", "Accessories"]

# dim_product_dupes: every 3rd product appears twice -> 4 duplicated ProductIDs.
dim_product_dupes = []
for index, product_id in enumerate(product_ids):
    category = CATEGORIES[index % len(CATEGORIES)]
    dim_product_dupes.append((product_id, f"Product {product_id}", category))
    if index % 3 == 0:
        dim_product_dupes.append((product_id, f"Product {product_id} (duplicate)", category))

# dim_promotion: 4 promotions with unique IDs.
dim_promotion = [
    (1, "Spring Sale", date(2008, 3, 1), date(2008, 3, 31)),
    (2, "Summer Clearance", date(2008, 6, 1), date(2008, 8, 31)),
    (3, "Back To School", date(2008, 9, 1), date(2008, 9, 30)),
    (4, "Holiday Bundle", date(2008, 11, 15), date(2008, 12, 31)),
]
promotion_ids = [row[0] for row in dim_promotion]

# bridge_product_promotion: unique (ProductID, PromotionID) pairs. Every 3rd product has
# two promotions, and every promotion covers several products.
pairs = set()
for index, product_id in enumerate(product_ids):
    pairs.add((product_id, promotion_ids[index % 4]))
    if index % 3 == 0:
        pairs.add((product_id, promotion_ids[(index + 1) % 4]))
bridge_product_promotion = sorted(pairs)

# dim_campaign: related to products only through the free-text ProductCategory.
# There is deliberately no mapping table.
dim_campaign = [
    (1, "Ride More", "Bikes"),
    (2, "Gear Up", "Accessories"),
    (3, "Build Your Own", "Components"),
    (4, "Wear It", "Clothing"),
]

# dim_date: one row per day, whole calendar years covering every OrderDate.
bounds = spark.sql(
    f"SELECT MIN(CAST(OrderDate AS date)) AS lo, MAX(CAST(OrderDate AS date)) AS hi "
    f"FROM {SOURCE_SCHEMA}.SalesLT_SalesOrderHeader"
).first()

if bounds["lo"] is None:
    raise RuntimeError(f"{SOURCE_SCHEMA}.SalesLT_SalesOrderHeader has no OrderDate values.")

day = date(bounds["lo"].year, 1, 1)
last_day = date(bounds["hi"].year, 12, 31)
dim_date = []
while day <= last_day:
    dim_date.append((
        day, day.year, f"Q{(day.month - 1) // 3 + 1}", day.month, calendar.month_name[day.month],
    ))
    day += timedelta(days=1)

In [ ]:
TABLES = {
    "dim_product_dupes": (dim_product_dupes, "ProductID int, Name string, Category string"),
    "dim_promotion": (dim_promotion, "PromotionID int, PromotionName string, StartDate date, EndDate date"),
    "bridge_product_promotion": (bridge_product_promotion, "ProductID int, PromotionID int"),
    "dim_campaign": (dim_campaign, "CampaignID int, CampaignName string, ProductCategory string"),
    "dim_date": (dim_date, "Date date, Year int, Quarter string, MonthNumber int, MonthName string"),
}

for name, (rows, ddl) in TABLES.items():
    (
        spark.createDataFrame(rows, ddl)
        .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(f"{SCHEMA}.{name}")
    )
    print(f"{SCHEMA}.{name}: {len(rows)} rows")

In [ ]:
def scalar(query):
    return spark.sql(query).first()[0]

duplicated = scalar(
    f"SELECT COUNT(*) FROM (SELECT ProductID FROM {SCHEMA}.dim_product_dupes "
    "GROUP BY ProductID HAVING COUNT(*) > 1)"
)
bridge_dupes = scalar(
    f"SELECT COUNT(*) FROM (SELECT ProductID, PromotionID FROM {SCHEMA}.bridge_product_promotion "
    "GROUP BY ProductID, PromotionID HAVING COUNT(*) > 1)"
)
orphans = scalar(
    f"SELECT COUNT(*) FROM {SCHEMA}.bridge_product_promotion b "
    f"LEFT JOIN {SOURCE_SCHEMA}.SalesLT_Product p ON b.ProductID = p.ProductID WHERE p.ProductID IS NULL"
)
uncovered = scalar(
    f"SELECT COUNT(*) FROM {SOURCE_SCHEMA}.SalesLT_SalesOrderHeader h "
    f"LEFT JOIN {SCHEMA}.dim_date d ON CAST(h.OrderDate AS date) = d.Date WHERE d.Date IS NULL"
)

print("duplicated ProductIDs:", duplicated, "(expect >= 3)")
print("duplicate bridge pairs:", bridge_dupes, "(expect 0)")
print("bridge ProductIDs not in saleslt:", orphans, "(expect 0)")
print("order dates missing from dim_date:", uncovered, "(expect 0)")

assert duplicated >= 3 and bridge_dupes == 0 and orphans == 0 and uncovered == 0, "Fixture checks failed"
print("All fixture checks passed.")